In [ ]:
import time
import datetime
import traceback
import threading
import queue
from dataclasses import dataclass, field
from pathlib import Path
from collections import deque

import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

import ipywidgets as widgets
from IPython.display import display

import winsound

from pypylon import pylon

# Ваши модули (не трогаем файлы)
from droplet_lib import DropletDetectorCuPy, detect_frame_to_dataset_layer
from Tracker import tracker_hook, get_last_tracks_layer, export_tracker_state, import_tracker_state, get_tracker_max_frame

import Grid_measurer as gm


In [ ]:
#%matplotlib widget
# хранить последние, например, 32 кадра (моно8)
_FRAMEBUF = deque(maxlen=32)  # list of (fid, gray)

In [ ]:
def sci_fmt(x, pos=None):
    if x == 0:
        return "0"
    exp = int(np.floor(np.log10(abs(x))))
    mant = x / 10**exp
    return rf"{mant:.1f}e{exp}"

def point_in_rotated_rect(px, py, p0, p1, half_width):
    """
    Проверка принадлежности точки (px,py) "ленточке" (прямоугольнику),
    заданной центральным сегментом p0->p1 и половинной шириной half_width.

    p0,p1: (x,y)
    """
    x0,y0 = float(p0[0]), float(p0[1])
    x1,y1 = float(p1[0]), float(p1[1])
    vx,vy = (x1-x0), (y1-y0)
    L2 = vx*vx + vy*vy
    if L2 < 1e-9:
        # вырожденный случай: считаем кругом радиуса half_width вокруг p0
        dx = px - x0
        dy = py - y0
        return (dx*dx + dy*dy) <= (half_width*half_width)

    # проекция на ось сегмента
    t = ((px-x0)*vx + (py-y0)*vy) / L2
    if t < 0.0 or t > 1.0:
        return False

    # расстояние до оси (перпендикуляр)
    projx = x0 + t*vx
    projy = y0 + t*vy
    dx = px - projx
    dy = py - projy
    return (dx*dx + dy*dy) <= (half_width*half_width)

def finish_rect_polygon(p0, p1, half_width):
    """
    4 вершины прямоугольника (x,y) вокруг центральной линии p0->p1.
    """
    x0,y0 = float(p0[0]), float(p0[1])
    x1,y1 = float(p1[0]), float(p1[1])
    vx,vy = (x1-x0), (y1-y0)
    L = (vx*vx + vy*vy) ** 0.5
    if L < 1e-9:
        # квадрат вокруг точки
        hw = float(half_width)
        return np.array([[x0-hw,y0-hw],[x0+hw,y0-hw],[x0+hw,y0+hw],[x0-hw,y0+hw]], dtype=np.float32)

    # единичный перпендикуляр
    nx, ny = -vy/L, vx/L
    hw = float(half_width)
    ox, oy = nx*hw, ny*hw

    a = (x0+ox, y0+oy)
    b = (x1+ox, y1+oy)
    c = (x1-ox, y1-oy)
    d = (x0-ox, y0-oy)
    return np.array([a,b,c,d], dtype=np.float32)


In [ ]:
# BGR цвета (OpenCV)
COLOR_RED    = (0, 0, 255)
COLOR_GREEN  = (0, 255, 0)
COLOR_GOLD   = (0, 215, 255)   # "золото" в BGR
COLOR_FINISH = (255, 0, 255)   # магента, чтобы не путать

@dataclass
class AppState:
    last_frame: np.ndarray | None = None  # mono8

    # последние данные от детектора/трекера (для перерисовки MPL без стрима)
    last_det_fid: int | None = None
    last_centers_yx: np.ndarray | None = None   # (N,2) int32 [y,x] full-res
    last_scores: np.ndarray | None = None       # (N,) float32
    last_tracks_layer: object | None = None     # то, что возвращает Tracker

    # finish-zone
    finish_enabled: bool = True
    finish_p0: tuple[int,int] = (300, 300)      # (x,y)
    finish_p1: tuple[int,int] = (1200, 320)     # (x,y)
    finish_width: int = 40                      # полуширина

    # favorite bubble
    favorite_enabled: bool = False
    favorite_id: int = -1
    favorite_in_finish_prev: bool = False  # чтобы орать только при входе

    # fake stream storage
    fake_frames: list[np.ndarray] | None = None
    fake_meta: dict = field(default_factory=dict)

    tracker_state: dict | None = None
    frame_base: int = 0
    last_global_fid: int = -1

    # background subtraction
    bg_enabled: bool = False               # включено ли вычитание
    bg_mean: np.ndarray | None = None      # (H,W) float32
    bg_nframes: int = 0                    # сколько кадров усредняли
    bg_timestamp: str = ""                 # когда снимали (для логов)
    bg_clip_to_uint8: bool = True          # клип к 0..255
    bg_use_renorm: bool = False            # опционально: растягивать после вычитания
    bg_renorm_p_lo: float = 1.0            # процентили для растяжки
    bg_renorm_p_hi: float = 99.0



STATE = AppState()

def tracks_to_items(tracks_layer):
    """
    ПРИМЕЧАНИЕ: это единственная функция-адаптер, которую возможно надо подстроить под твой Tracker.

    Возвращает список элементов:
        items = [{"id": int, "y": float, "x": float, ...}, ...]
    """
    if tracks_layer is None:
        return []

    # 1) numpy array (N, >=3): [id, y, x] или [y,x,id] — попробуем угадать
    if isinstance(tracks_layer, np.ndarray):
        arr = tracks_layer
        if arr.ndim == 2 and arr.shape[1] >= 3:
            # эвристика: если первый столбец похоже на int id (маленькие)
            c0 = arr[:,0]
            c2 = arr[:,2]
            if np.all(np.isfinite(c0)) and np.all((c0 == np.floor(c0))):
                # считаем [id, y, x]
                out = []
                for r in arr:
                    out.append({"id": int(r[0]), "y": float(r[1]), "x": float(r[2])})
                return out
            elif np.all(np.isfinite(c2)) and np.all((c2 == np.floor(c2))):
                # считаем [y, x, id]
                out = []
                for r in arr:
                    out.append({"id": int(r[2]), "y": float(r[0]), "x": float(r[1])})
                return out

    # 2) list of dicts or list of tuples
    if isinstance(tracks_layer, (list, tuple)):
        if len(tracks_layer) == 0:
            return []
        # list of dicts
        if isinstance(tracks_layer[0], dict):
            out = []
            for d in tracks_layer:
                if "id" in d and ("y" in d and "x" in d):
                    out.append({"id": int(d["id"]), "y": float(d["y"]), "x": float(d["x"])})
            return out
        # list of tuples (id,y,x) or (y,x,id)
        if isinstance(tracks_layer[0], (list, tuple)) and len(tracks_layer[0]) >= 3:
            out = []
            # эвристика по целочисленности
            a0 = [t[0] for t in tracks_layer]
            if all(float(v).is_integer() for v in a0):
                for t in tracks_layer:
                    out.append({"id": int(t[0]), "y": float(t[1]), "x": float(t[2])})
                return out
            else:
                for t in tracks_layer:
                    out.append({"id": int(t[2]), "y": float(t[0]), "x": float(t[1])})
                return out

    # 3) если Tracker возвращает объект с .tracks или .active_tracks
    for attr in ("tracks", "active_tracks"):
        if hasattr(tracks_layer, attr):
            val = getattr(tracks_layer, attr)
            return tracks_to_items(val)

    return []


In [ ]:
def draw_centers_on_frame(img, centers_yx, color=COLOR_RED, markerSize=20, thickness=2):
    """
    img: gray (H,W) uint8 or BGR
    centers_yx: (N,2) [y,x]
    """
    if img.ndim == 2:
        disp = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    else:
        disp = img

    if centers_yx is None:
        return disp

    for (y, x) in centers_yx:
        cv2.drawMarker(
            disp, (int(x), int(y)), color,
            markerType=cv2.MARKER_CROSS,
            markerSize=int(markerSize),
            thickness=int(thickness),
            line_type=cv2.LINE_AA
        )
    return disp

def draw_tracks_on_frame(img, tracks_layer, favorite_id=None,
                         color=COLOR_GREEN, fav_color=COLOR_GOLD):
    if tracks_layer is None:
        return img

    # img должен быть BGR, но на всякий случай:
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    arr = np.asarray(tracks_layer)
    if arr.size == 0:
        return img
    if arr.ndim != 2 or arr.shape[1] < 4:
        return img

    H, W = img.shape[:2]
    fav = None if favorite_id is None else int(favorite_id)

    for row in arr:
        # row: [frame, id, y, x, score]
        tid = int(row[1])
        y = float(row[2]); x = float(row[3])

        if not np.isfinite(x) or not np.isfinite(y):
            continue
        xi = int(round(x)); yi = int(round(y))
        if xi < 0 or xi >= W or yi < 0 or yi >= H:
            continue

        is_fav = (fav is not None and tid == fav)
        col = fav_color if is_fav else color
        ms  = 26 if is_fav else 18
        th  = 3  if is_fav else 2

        cv2.drawMarker(img, (xi, yi), col,
                       markerType=cv2.MARKER_CROSS,
                       markerSize=ms, thickness=th, line_type=cv2.LINE_AA)

        # Текст с обводкой (чёрной)
        org = (xi + 8, yi - 8)
        txt = f"{tid}"
        cv2.putText(img, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(img, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.7, col,      2, cv2.LINE_AA)

    return img


def draw_finish_zone(img, p0, p1, half_width):
    """
    Рисует повёрнутый прямоугольник finish-зоны и центральную линию.
    p0,p1: (x,y)
    half_width: int
    """
    if img.ndim == 2:
        disp = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    else:
        disp = img

    poly = finish_rect_polygon(p0, p1, half_width)  # (4,2) float
    pts = poly.reshape(-1,1,2).astype(np.int32)

    # заливка полупрозрачная
    overlay = disp.copy()
    cv2.fillPoly(overlay, [pts], COLOR_FINISH)
    disp = cv2.addWeighted(overlay, 0.25, disp, 0.75, 0.0)

    # контур + центральная линия
    cv2.polylines(disp, [pts], isClosed=True, color=COLOR_FINISH, thickness=2, lineType=cv2.LINE_AA)
    cv2.line(disp, tuple(map(int,p0)), tuple(map(int,p1)), COLOR_FINISH, 2, cv2.LINE_AA)
    return disp

def check_favorite_cross_finish(tracks_layer):
    """
    tracks_layer: (N,5) float32: [frame, id, y, x, score]
    """
    if not (STATE.favorite_enabled and STATE.finish_enabled):
        STATE.favorite_in_finish_prev = False
        return

    fav = int(STATE.favorite_id)
    arr = np.asarray(tracks_layer)
    if arr.size == 0 or arr.ndim != 2 or arr.shape[1] < 4:
        return

    # ищем строку избранного id
    ids = arr[:, 1].astype(np.int32, copy=False)
    m = (ids == fav)
    if not np.any(m):
        return

    row = arr[m][0]
    py = float(row[2])
    px = float(row[3])

    in_zone = point_in_rotated_rect(px, py, STATE.finish_p0, STATE.finish_p1, STATE.finish_width)

    if in_zone and not STATE.favorite_in_finish_prev:
        try:
            winsound.Beep(2000, 250)
            winsound.Beep(2400, 250)
        except Exception:
            pass
        with out:
            print(f"🚩 FAVORITE #{fav} entered finish zone!")

    STATE.favorite_in_finish_prev = bool(in_zone)


def make_disp(gray_u8: np.ndarray, colorize: bool) -> np.ndarray:
    """
    Возвращает BGR картинку для отрисовки оверлеев.
    """
    if colorize:
        return cv2.applyColorMap(gray_u8, cv2.COLORMAP_TURBO)
    return cv2.cvtColor(gray_u8, cv2.COLOR_GRAY2BGR)

def apply_background_subtraction(gray_u8: np.ndarray) -> np.ndarray:
    """
    gray_u8: mono8 (H,W) uint8
    returns: mono8 uint8 after (gray - bg_mean)
    """
    if (not STATE.bg_enabled) or (STATE.bg_mean is None):
        return gray_u8

    # float32 subtraction
    out = gray_u8.astype(np.float32) - STATE.bg_mean

    # clip negatives
    if STATE.bg_clip_to_uint8:
        out = np.clip(out, 0.0, 255.0)

    if STATE.bg_use_renorm:
        # robust stretch to 0..255 by percentiles
        lo = float(np.percentile(out, STATE.bg_renorm_p_lo))
        hi = float(np.percentile(out, STATE.bg_renorm_p_hi))
        if hi > lo + 1e-6:
            out = (out - lo) * (255.0 / (hi - lo))
        out = np.clip(out, 0.0, 255.0)

    return out.astype(np.uint8, copy=False)



In [ ]:
class DetectorWorker:
    """
    Threaded detector wrapper:
    - in_q keeps only the newest frame (maxsize=1)
    - out_q keeps small amount of results
    """

    def __init__(self):
        self.detector = None
        self.enabled = False

        self._stop = threading.Event()
        self._enabled_evt = threading.Event()

        self.in_q = queue.Queue(maxsize=1)
        self.out_q = queue.Queue(maxsize=2)

        self._warmup_left = 0
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

        self.last_result = None  # (fid, centers_yx, scores)
        self.last_layer = None   # (fid, layer)

    def set_detector(self, detector):
        self.detector = detector

    def set_warmup(self, n: int):
        self._warmup_left = int(n)

    def start(self):
        self.enabled = True
        self._enabled_evt.set()

    def stop(self):
        self.enabled = False
        self._enabled_evt.clear()

    def shutdown(self):
        self._stop.set()
        self._enabled_evt.set()
        try:
            self._thread.join(timeout=1.0)
        except Exception:
            pass

    def push_frame(self, frame_id: int, gray: np.ndarray):
        if not self.enabled or self.detector is None:
            return
        item = (int(frame_id), gray.copy())
        if self.in_q.full():
            try:
                self.in_q.get_nowait()
            except queue.Empty:
                pass
        try:
            self.in_q.put_nowait(item)
        except queue.Full:
            pass

    def _drain_out(self):
        while True:
            try:
                fid, centers, scores, layer = self.out_q.get_nowait()
                self.last_result = (fid, centers, scores)
                self.last_layer = (fid, layer)
            except queue.Empty:
                break

    def get_latest(self):
        self._drain_out()
        return self.last_result  # or None

    def _loop(self):
        while not self._stop.is_set():
            self._enabled_evt.wait(timeout=0.1)
            if self._stop.is_set():
                break
            if not self.enabled or self.detector is None:
                continue

            try:
                fid, gray = self.in_q.get(timeout=0.1)
            except queue.Empty:
                continue

            try:
                layer = detect_frame_to_dataset_layer(gray, fid, self.detector)  # (N,4): fid,y,x,score
            except Exception:
                layer = np.empty((0, 4), dtype=np.float32)

            if layer.shape[0] == 0:
                centers = np.empty((0, 2), dtype=np.int32)
                scores  = np.empty((0,), dtype=np.float32)
            else:
                centers = layer[:, 1:3].astype(np.int32, copy=True)
                scores  = layer[:, 3].astype(np.float32, copy=True)

            if self._warmup_left > 0:
                self._warmup_left -= 1
                continue

            if self.out_q.full():
                try:
                    self.out_q.get_nowait()
                except queue.Empty:
                    pass
            try:
                self.out_q.put_nowait((fid, centers, scores, layer))
            except queue.Full:
                pass

DET = DetectorWorker()


In [ ]:
def make_camera():
    cam = pylon.InstantCamera(pylon.TlFactory.GetInstance().CreateFirstDevice())
    conv = pylon.ImageFormatConverter()
    conv.OutputPixelFormat = pylon.PixelType_Mono8
    conv.OutputBitAlignment = pylon.OutputBitAlignment_MsbAligned
    return cam, conv

def camera_configure(cam, fps=None, exposure_ms=None):
    cam.Open()
    if exposure_ms is not None:
        cam.ExposureTime.Value = float(exposure_ms) * 1000.0  # ms->us
    if fps is not None:
        cam.AcquisitionFrameRateEnable.Value = True
        cam.AcquisitionFrameRate.Value = float(fps)

def iter_frames_real(fps=None, exposure_ms=None, timeout_ms=5000):
    cam, conv = make_camera()
    try:
        camera_configure(cam, fps=fps, exposure_ms=exposure_ms)
        cam.StartGrabbing()
        frame_id = 0
        while cam.IsGrabbing():
            grab = cam.RetrieveResult(timeout_ms, pylon.TimeoutHandling_ThrowException)
            try:
                if grab.GrabSucceeded():
                    img = conv.Convert(grab).GetArray()
                    yield frame_id, img
                    frame_id += 1
            finally:
                grab.Release()
    finally:
        try: cam.StopGrabbing()
        except Exception: pass
        try: cam.Close()
        except Exception: pass

def iter_frames_fake(frames: list[np.ndarray], loop=True, target_fps=20.0):
    if target_fps <= 0:
        target_fps = 20.0
    dt = 1.0 / target_fps
    t_next = time.perf_counter()

    idx = 0
    n = len(frames)
    frame_id = 0
    while True:
        gray = frames[idx]
        yield frame_id, gray
        frame_id += 1

        # pacing
        t_now = time.perf_counter()
        if t_now < t_next:
            time.sleep(t_next - t_now)
        t_next += dt

        idx += 1
        if idx >= n:
            if loop:
                idx = 0
            else:
                break

def build_overlay_from_state(colorize: bool):
    if STATE.last_frame is None:
        return None

    gray = STATE.last_frame
    disp = make_disp(gray, colorize)

    if STATE.finish_enabled:
        disp = draw_finish_zone(disp, STATE.finish_p0, STATE.finish_p1, int(STATE.finish_width))

    if STATE.last_tracks_layer is not None:
        fav = STATE.favorite_id if STATE.favorite_enabled else None
        disp = draw_tracks_on_frame(disp, STATE.last_tracks_layer, favorite_id=fav)

    # centers отдельно можно не рисовать, если draw_tracks рисует крестики,
    # но если хочешь — оставь
    if STATE.last_centers_yx is not None:
        disp = draw_centers_on_frame(disp, STATE.last_centers_yx, color=COLOR_RED)

    return disp


def _get_frame_from_buf(det_fid: int):
    # ищем с конца (самое частое попадание)
    for f, g in reversed(_FRAMEBUF):
        if f == det_fid:
            return g
    return None


In [ ]:
def run_preview(frame_iter, *, colorize=True, window_name="Preview (q=close)",
                record_path: str | None = None,
                record_fps: float = 20.0,
                record_fourcc: str = "MJPG",
                record_seconds: float | None = None):

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 1000, 700)

    writer = None
    t_end = None
    if record_path is not None and record_seconds is not None:
        t_end = time.perf_counter() + float(record_seconds)

    try:
        for fid, gray in frame_iter:
            fid = int(fid)
            STATE.last_frame = gray

            # BG state -> apply subtraction
            apply_bg_from_widgets()
            gray = apply_background_subtraction(gray)

            _FRAMEBUF.append((fid, gray))
            DET.push_frame(fid, gray)

            # базовый кадр всегда BGR
            disp = make_disp(gray, bool(colorize))

            res = DET.get_latest()
            if res is not None:
                det_fid, centers, scores = res
                det_fid = int(det_fid)

                det_gray = _get_frame_from_buf(det_fid)
                if det_gray is None:
                    det_gray = gray

                disp = make_disp(det_gray, bool(colorize))

                tracker_hook(det_fid, centers, scores)
                tracks_layer = get_last_tracks_layer()

                check_favorite_cross_finish(tracks_layer)

                if STATE.finish_enabled:
                    disp = draw_finish_zone(disp, STATE.finish_p0, STATE.finish_p1, int(STATE.finish_width))

                fav = STATE.favorite_id if STATE.favorite_enabled else None
                disp = draw_tracks_on_frame(disp, tracks_layer, favorite_id=fav)

                # сохранить для MPL
                STATE.last_det_fid = det_fid
                STATE.last_centers_yx = None if centers is None else centers.copy()
                STATE.last_scores = None if scores is None else scores.copy()
                STATE.last_tracks_layer = None if tracks_layer is None else np.asarray(tracks_layer).copy()
                STATE.last_overlay = disp  # BGR

            # --- запись AVI (пишем именно disp с оверлеями) ---
            if record_path is not None:
                if writer is None:
                    h, w = disp.shape[:2]
                    fourcc = cv2.VideoWriter_fourcc(*record_fourcc)
                    writer = cv2.VideoWriter(str(record_path), fourcc, float(record_fps), (w, h), True)
                    if not writer.isOpened():
                        ui_log(f"⚠️ Failed to open VideoWriter: {record_path}")
                        writer = None
                    else:
                        ui_log(f"🎥 Recording AVI → {record_path}")

                if writer is not None:
                    writer.write(disp)

            cv2.imshow(window_name, disp)

            k = cv2.waitKey(1) & 0xFF
            if k == ord("q"):
                break

            # авто-стоп по времени записи
            if t_end is not None and time.perf_counter() >= t_end:
                ui_log("⏹ AVI record: duration reached, stopping preview")
                break

    finally:
        if writer is not None:
            writer.release()
            ui_log("✅ AVI saved")
        cv2.destroyAllWindows()
        try:
            winsound.Beep(800, 150)
        except Exception:
            pass


In [ ]:
def capture_background_from_camera(_=None):
    """
    Снимает N кадров с РЕАЛЬНОЙ камеры, усредняет, кладёт в STATE.bg_mean (float32).
    """
    apply_bg_from_widgets()
    N = int(bg_n_w.value)
    if N <= 0:
        ui_log("⚠️ BG N must be > 0")
        return

    cam, conv = make_camera()
    try:
        camera_configure(cam, fps=None, exposure_ms=float(exposure_w.value))
        cam.StartGrabbing()

        acc = None
        got = 0
        t0 = time.time()

        ui_log(f"📸 Capturing background: N={N} ...")
        while cam.IsGrabbing() and got < N:
            grab = cam.RetrieveResult(5000, pylon.TimeoutHandling_ThrowException)
            try:
                if grab.GrabSucceeded():
                    img = conv.Convert(grab).GetArray()  # mono8
                    if acc is None:
                        acc = img.astype(np.float32)
                    else:
                        acc += img.astype(np.float32)
                    got += 1
            finally:
                grab.Release()

        if acc is None or got == 0:
            ui_log("⚠️ Failed to capture background (no frames)")
            return

        bg = acc / float(got)

        STATE.bg_mean = bg.astype(np.float32, copy=False)
        STATE.bg_nframes = int(got)
        STATE.bg_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        ui_log(f"✅ Background captured: {got} frames, shape={STATE.bg_mean.shape}, t={(time.time()-t0):.2f}s")
        try:
            winsound.Beep(1800, 150)
        except Exception:
            pass

    except Exception:
        traceback.print_exc()
        ui_log("⚠️ Background capture failed (see traceback)")
    finally:
        try:
            cam.StopGrabbing()
        except Exception:
            pass
        try:
            cam.Close()
        except Exception:
            pass

    # обновим MPL превью (покажет уже фоновычтенный кадр)
    update_crop_preview()


In [ ]:
def record_sequence(sample_name: str, fps: float, exposure_ms: float, duration_s: float,
                    crop: tuple[int,int,int,int], day_dir: Path):
    x1, x2, y1, y2 = crop
    x1, x2 = sorted([int(x1), int(x2)])
    y1, y2 = sorted([int(y1), int(y2)])

    n_frames = int(duration_s * fps)
    timestamps = []
    brightness = []

    cam, conv = make_camera()
    win = "Live view (REC) (q=stop)"
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, 1000, 700)

    try:
        camera_configure(cam, fps=fps, exposure_ms=exposure_ms)
        cam.StartGrabbing()

        print(f"▶ Recording {n_frames} frames @ {fps:g} fps")

        for i in range(n_frames):
            grab = cam.RetrieveResult(5000, pylon.TimeoutHandling_ThrowException)
            try:
                if grab.GrabSucceeded():
                    img = conv.Convert(grab).GetArray()
                    STATE.last_frame = img.copy()

                    cropped = img[y1:y2, x1:x2]
                    brightness.append(np.sum(cropped, dtype=np.int64))
                    timestamps.append(grab.TimeStamp)

                    disp = cv2.applyColorMap(img, cv2.COLORMAP_TURBO)
                    text = f"Frame {i+1}/{n_frames} ({(i+1)/n_frames*100:.1f}%)"
                    cv2.putText(disp, text, (80, 120), cv2.FONT_HERSHEY_SIMPLEX,
                                2.0, (255, 255, 255), 2, cv2.LINE_AA)
                    cv2.imshow(win, disp)

                    if (cv2.waitKey(1) & 0xFF) == ord("q"):
                        break
            finally:
                grab.Release()

    except Exception:
        traceback.print_exc()
    finally:
        try: cam.StopGrabbing()
        except Exception: pass
        try: cam.Close()
        except Exception: pass
        cv2.destroyAllWindows()
        try: winsound.Beep(1200, 300)
        except Exception: pass

    # save outputs
    day_dir.mkdir(parents=True, exist_ok=True)
    ts_file = day_dir / f"{sample_name}_timestamps.txt"
    br_file = day_dir / f"{sample_name}_brightness.txt"
    md_file = day_dir / f"{sample_name}_metadata.txt"

    np.savetxt(ts_file, np.column_stack((np.arange(len(timestamps)), np.array(timestamps, dtype=np.int64))), fmt="%d\t%d")
    np.savetxt(br_file, np.column_stack((np.arange(len(brightness)), np.array(brightness, dtype=np.float64))), fmt="%d\t%.1f")

    with open(md_file, "w", encoding="utf-8") as f:
        f.write(f"FPS={fps}\nExposure_ms={exposure_ms}\nDuration_s={duration_s}\nCrop={crop}\n")

    print("✅ Recording finished")
    return ts_file, br_file, md_file

def plot_last_experiment(ts_file: Path, br_file: Path):
    t = np.loadtxt(ts_file)[:, 1]
    t = (t - t[0]) / 1e9
    b = np.loadtxt(br_file)[:, 1]
    length = len(b)
    t_cr = t[length//20:]
    b_cr = b[length//20:]

    plt.figure(figsize=(6, 3))
    plt.plot(t_cr, b_cr)
    plt.xlabel("Time (s)")
    plt.ylabel("Brightness")
    plt.grid(True)
    plt.show()

def crop_preview_plot(x1, x2, y1, y2, *, colorize=True):
    if STATE.last_frame is None:
        print("⚠️ No last frame yet")
        return

    x1, x2 = sorted([int(x1), int(x2)])
    y1, y2 = sorted([int(y1), int(y2)])

    frame_raw = STATE.last_frame
    cropped = frame_raw[y1:y2, x1:x2]
    crop_sum = cropped.sum(dtype=np.int64)

    # <-- ключевая часть: пересобираем оверлей прямо сейчас
    frame_show = build_overlay_from_state(colorize=colorize)
    if frame_show is None:
        frame_show = frame_raw

    fig = plt.figure(figsize=(6, 4))
    gs = fig.add_gridspec(1, 2, width_ratios=[3, 2], wspace=0.25)

    ax_img = fig.add_subplot(gs[0, 0])
    ax_hist = fig.add_subplot(gs[0, 1])

    if frame_show.ndim == 3:
        ax_img.imshow(cv2.cvtColor(frame_show, cv2.COLOR_BGR2RGB))
    else:
        ax_img.imshow(frame_show, cmap="gray")

    # crop рамка
    ax_img.axvline(x1); ax_img.axvline(x2)
    ax_img.axhline(y1); ax_img.axhline(y2)

    ax_img.set_title("Crop + overlay", fontsize=9)
    ax_img.tick_params(labelsize=7)

    ax_hist.hist(cropped.ravel(), bins=50)
    ax_hist.set_title(rf"$\Sigma={sci_fmt(crop_sum)}$", fontsize=9)
    ax_hist.tick_params(labelsize=7)
    ax_hist.xaxis.set_major_formatter(FuncFormatter(sci_fmt))
    ax_hist.yaxis.set_major_formatter(FuncFormatter(sci_fmt))

    for ax in (ax_img, ax_hist):
        ax.set_anchor("N")
        ax.margins(x=0.02, y=0.05)

    plt.show()

def save_preview_frame(folder: Path, fname: str):
    if STATE.last_frame is None:
        raise RuntimeError("No last frame to save")
    folder.mkdir(parents=True, exist_ok=True)

    img_path = folder / fname
    hist_path = folder / (Path(fname).stem + "_hist.png")
    txt_path = folder / (Path(fname).stem + "_hist.txt")

    cv2.imwrite(str(img_path), STATE.last_frame)

    s = STATE.last_frame.sum(dtype=np.int64)
    plt.figure(figsize=(6, 4))
    plt.hist(STATE.last_frame.ravel(), bins=50)
    plt.title(f"{fname}\nΣ = {s:.0f}")
    plt.xlabel("Intensity")
    plt.ylabel("Counts")
    plt.tight_layout()
    plt.savefig(hist_path, dpi=150)
    plt.close()

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"Sum = {s}\n")

    return img_path, hist_path, txt_path

def show_hist_from_saved(folder: Path, fname: str):
    img_path = folder / fname
    if not img_path.exists():
        raise FileNotFoundError(str(img_path))
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise RuntimeError("Failed to load image")
    s = img.sum(dtype=np.int64)
    plt.figure(figsize=(6, 4))
    plt.hist(img.ravel(), bins=50)
    plt.title(f"{fname}\nΣ = {s:.0f}")
    plt.xlabel("Intensity")
    plt.ylabel("Counts")
    plt.tight_layout()
    plt.show()


In [ ]:
out = widgets.Output()

def ui_log(msg: str):
    with out:
        print(msg)

# Core params
fps_w      = widgets.FloatText(20.0, description="FPS")
exposure_w = widgets.FloatText(15.0, description="Exposure ms")
duration_w = widgets.FloatText(10.0, description="Duration s")
laser_w    = widgets.FloatText(1.0, description="Laser")

root_w   = widgets.Text("", description="Folder")
sample_w = widgets.Text("", description="Sample")

# Video / fake stream
video_path_w = widgets.Text("", description="Video file", layout=widgets.Layout(width="800px"))
use_fake_w   = widgets.Checkbox(False, description="Use RAM stream")
loop_fake_w  = widgets.Checkbox(True,  description="Loop")
colorize_w   = widgets.Checkbox(True,  description="Colorize preview")

preload_btn   = widgets.Button(description="📥 Preload video to RAM", button_style="info")
fake_prev_btn = widgets.Button(description="▶ Open preview", button_style="success")
real_prev_btn = widgets.Button(description="👁 Live view", button_style="info")

# Detector controls
det_enable_w = widgets.Checkbox(False, description="Detector ON")
det_warmup_w = widgets.IntText(5, description="Warmup frames")

det_backend_w = widgets.Dropdown(options=["fft","ndimage"], value="fft", description="corr_backend")
det_ds_w      = widgets.IntText(4, description="downscale")
det_peak_ds_w = widgets.IntText(2, description="peak_downscale")
det_pct_w     = widgets.FloatText(88.0, description="percentile")
det_min_dist_w= widgets.IntText(40, description="min_dist")
det_radius_w  = widgets.IntText(100, description="radius_px")

det_thr_mode_w  = widgets.Dropdown(options=["per_frame","cached","ema"], value="cached", description="thr_mode")
det_thr_every_w = widgets.IntText(500, description="thr_every")
det_thr_alpha_w = widgets.FloatText(0.1, description="thr_ema_a")
det_gpu_down_w  = widgets.Checkbox(False, description="GPU downscale")
det_stride_w    = widgets.IntText(1, description="pct_stride")

det_init_btn   = widgets.Button(description="Init detector", button_style="info")
det_toggle_btn = widgets.Button(description="Start/Stop detector", button_style="success")

# Favorite bubble controls
fav_enable_w = widgets.Checkbox(False, description="Favorite ON")
fav_id_w     = widgets.IntText(-1, description="Favorite ID")

# Crop sliders
H, W = (STATE.last_frame.shape if STATE.last_frame is not None else (2178, 3860))
x1_w = widgets.IntSlider(0, 0, W-1, description="x1")
x2_w = widgets.IntSlider(W-1, 0, W-1, description="x2")
y1_w = widgets.IntSlider(0, 0, H-1, description="y1")
y2_w = widgets.IntSlider(H-1, 0, H-1, description="y2")

setdef_btn = widgets.Button(description="Set default crop", button_style="info", icon="compress")
full_btn   = widgets.Button(description="Full frame", button_style="warning", icon="expand")

# Finish zone controls (x,y endpoints + half-width)
finish_on_w  = widgets.Checkbox(True, description="Finish ON")
fx0_w = widgets.IntSlider(300, 0, W-1, description="fx0")
fy0_w = widgets.IntSlider(300, 0, H-1, description="fy0")
fx1_w = widgets.IntSlider(1200, 0, W-1, description="fx1")
fy1_w = widgets.IntSlider(320, 0, H-1, description="fy1")
fwidth_w = widgets.IntSlider(40, 1, 400, description="halfW")

# Record & save
start_rec_btn = widgets.Button(description="▶ Start recording", button_style="success")
save_name_w   = widgets.Text("preview_frame.png", description="Save frame as:")
save_btn      = widgets.Button(description="💾 Save preview frame")
show_hist_btn = widgets.Button(description="📊 Show hist from file")

# Background controls
bg_on_w = widgets.Checkbox(False, description="BG subtract ON")
bg_n_w  = widgets.IntText(50, description="BG N frames")
bg_cap_btn = widgets.Button(description="📸 Capture background", button_style="warning")

bg_renorm_w = widgets.Checkbox(False, description="renorm after BG")
bg_p_lo_w = widgets.FloatText(1.0, description="p_lo")
bg_p_hi_w = widgets.FloatText(99.0, description="p_hi")

# --- AVI recording controls ---
avi_on_w     = widgets.Checkbox(False, description="Record AVI")
avi_sec_w    = widgets.FloatText(10.0, description="AVI seconds")
avi_name_w   = widgets.Text("preview_overlay.avi", description="AVI filename")
avi_fps_w    = widgets.FloatText(0.0, description="AVI fps (0=use FPS)")
avi_fourcc_w = widgets.Dropdown(options=["MJPG", "XVID"], value="MJPG", description="fourcc")


def apply_finish_from_widgets():
    STATE.finish_enabled = bool(finish_on_w.value)
    STATE.finish_p0 = (int(fx0_w.value), int(fy0_w.value))
    STATE.finish_p1 = (int(fx1_w.value), int(fy1_w.value))
    STATE.finish_width = int(fwidth_w.value)

def apply_favorite_from_widgets():
    STATE.favorite_enabled = bool(fav_enable_w.value)
    STATE.favorite_id = int(fav_id_w.value)
    # сброс "внутри" чтобы событие входа ловилось корректно после смены id
    STATE.favorite_in_finish_prev = False

def update_crop_preview(_=None):
    apply_finish_from_widgets()
    apply_favorite_from_widgets()
    with out:
        out.clear_output(wait=True)
        crop_preview_plot(x1_w.value, x2_w.value, y1_w.value, y2_w.value, colorize=bool(colorize_w.value))

def set_default_crop(_=None):
    x1_w.value = 1000
    x2_w.value = 3000
    y1_w.value = 500
    y2_w.value = 2000

def set_full_crop(_=None):
    x1_w.value = 0
    x2_w.value = W - 1
    y1_w.value = 0
    y2_w.value = H - 1

def init_detector(_=None):
    detector = DropletDetectorCuPy(
        percentile=float(det_pct_w.value),
        downscale_factor=int(det_ds_w.value),
        droplet_radius_px=int(det_radius_w.value),
        min_distance=int(det_min_dist_w.value),
        cpu_downscale=not bool(det_gpu_down_w.value),
        sample_percentile_stride=int(det_stride_w.value),
        max_candidates=20000,
        corr_backend=str(det_backend_w.value),
        peak_downscale=int(det_peak_ds_w.value),
        thr_mode=str(det_thr_mode_w.value),
        thr_update_every=int(det_thr_every_w.value),
        thr_ema_alpha=float(det_thr_alpha_w.value),
        template_path=None,
        template_array=None,
        template_resize="auto",
    )
    DET.set_detector(detector)
    DET.set_warmup(int(det_warmup_w.value))

    if det_enable_w.value:
        DET.start()

    ui_log("✅ Detector initialized")

def get_avi_config():
    """
    Возвращает (enabled, path, max_seconds, fps, fourcc) или (False, None, 0, 0, 'MJPG')
    """
    enabled = bool(avi_on_w.value)
    if not enabled:
        return False, None, 0.0, 0.0, "MJPG"

    fname = avi_name_w.value.strip().strip('"')
    if fname == "":
        ui_log("⚠️ AVI filename is empty")
        return False, None, 0.0, 0.0, str(avi_fourcc_w.value)

    sec = float(avi_sec_w.value)
    if sec <= 0:
        ui_log("⚠️ AVI seconds must be > 0")
        return False, None, 0.0, 0.0, str(avi_fourcc_w.value)

    root = Path(root_w.value.strip().strip('"')) if root_w.value.strip() else None
    if root is None or not root.exists():
        ui_log("⚠️ Folder (root_w) must exist to save AVI")
        return False, None, 0.0, 0.0, str(avi_fourcc_w.value)

    out_path = root / fname

    # fps: если 0 — берём из fps_w
    fps_file = float(avi_fps_w.value)
    if fps_file <= 0:
        fps_file = float(fps_w.value)

    fourcc = str(avi_fourcc_w.value)
    return True, str(out_path), sec, fps_file, fourcc


def apply_detector_enable(_=None):
    if det_enable_w.value:
        DET.set_warmup(int(det_warmup_w.value))
        DET.start()
        ui_log("🟢 Detector thread enabled")
    else:
        DET.stop()
        ui_log("🔴 Detector thread disabled")

def toggle_detector_btn(_=None):
    det_enable_w.value = not det_enable_w.value

def preload_video(_=None):
    path = video_path_w.value.strip().strip('"')
    if path == "":
        ui_log("⚠️ Video file path is empty")
        return

    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        ui_log(f"⚠️ Cannot open video: {path}")
        return

    frames = []
    t0 = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame.ndim == 3:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        else:
            gray = frame
        if gray.dtype != np.uint8:
            gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        frames.append(gray)

    cap.release()

    if not frames:
        ui_log("⚠️ No frames read from video")
        return

    STATE.fake_frames = frames
    STATE.fake_meta = {"path": path, "n_frames": len(frames), "shape": frames[0].shape, "load_s": time.time() - t0}
    STATE.last_frame = frames[0].copy()

    ui_log(f"✅ Preloaded {STATE.fake_meta['n_frames']} frames; shape={STATE.fake_meta['shape']}; load_s={STATE.fake_meta['load_s']:.2f}")
    update_crop_preview()

def start_fake_preview(_=None):
    if not use_fake_w.value:
        ui_log("⚠️ Enable 'Use RAM stream' first")
        return
    if not STATE.fake_frames:
        ui_log("⚠️ No preloaded frames. Click 'Preload video to RAM' first.")
        return

    apply_finish_from_widgets()
    apply_favorite_from_widgets()

    # --- restore tracker state (if any) ---
    try:
        if getattr(STATE, "tracker_state", None) is not None:
            import_tracker_state(STATE.tracker_state)
    except Exception:
        pass

    # --- advance global frame numbering so IDs don't explode on restart ---
    # continue after the last processed global frame
    STATE.frame_base = int(getattr(STATE, "last_global_fid", -1) + 1)

    ui_log(f"▶ Show preview started: n={len(STATE.fake_frames)}, loop={loop_fake_w.value}, fps={fps_w.value:g}")
    rec_on, rec_path, rec_sec, rec_fps, rec_fourcc = get_avi_config()
    if not rec_on:
        rec_path = None
        rec_sec = None

    it = iter_frames_fake(STATE.fake_frames, loop=loop_fake_w.value, target_fps=float(fps_w.value))
    run_preview(
        it,
        colorize=bool(colorize_w.value),
        window_name="RAM stream preview (q=close)",
        record_path=rec_path,
        record_fps=rec_fps,
        record_fourcc=rec_fourcc,
        record_seconds=rec_sec
    )


def start_real_preview(_=None):
    apply_finish_from_widgets()
    apply_favorite_from_widgets()

    # --- restore tracker state (if any) ---
    try:
        if getattr(STATE, "tracker_state", None) is not None:
            import_tracker_state(STATE.tracker_state)
    except Exception:
        pass

    # --- keep global frame id monotonic across restarts ---
    STATE.frame_base = int(getattr(STATE, "last_global_fid", -1) + 1)

    ui_log("▶ Live preview started (q to close)")

    rec_on, rec_path, rec_sec, rec_fps, rec_fourcc = get_avi_config()

    if not rec_on:
        rec_path = None
        rec_sec = None

    it = iter_frames_real(fps=None, exposure_ms=float(exposure_w.value))
    run_preview(
        it,
        colorize=bool(colorize_w.value),
        window_name="Live preview (q=close)",
        record_path=rec_path,
        record_fps=rec_fps,
        record_fourcc=rec_fourcc,
        record_seconds=rec_sec
    )



def start_recording(_=None):
    folder = Path(root_w.value.strip().strip('"'))
    sample = sample_w.value.strip()
    if not folder.exists():
        ui_log("⚠️ Folder does not exist")
        return
    if sample == "":
        ui_log("⚠️ Sample name is empty")
        return

    today = datetime.date.today().strftime("%Y-%m-%d")
    day_dir = folder / f"{today}_experiments"
    day_dir.mkdir(parents=True, exist_ok=True)

    crop = (x1_w.value, x2_w.value, y1_w.value, y2_w.value)
    ui_log("▶ Recording started")
    ts_file, br_file, md_file = record_sequence(sample, float(fps_w.value), float(exposure_w.value), float(duration_w.value),
                                                crop=crop, day_dir=day_dir)

    with out:
        print("✅ Recording finished")
        plot_last_experiment(ts_file, br_file)

def on_save_frame(_=None):
    folder = Path(root_w.value.strip().strip('"'))
    if not folder.exists():
        ui_log("⚠️ Root folder does not exist")
        return
    fname = save_name_w.value.strip()
    if fname == "":
        ui_log("⚠️ Empty filename")
        return
    try:
        img_path, hist_path, txt_path = save_preview_frame(folder, fname)
        ui_log(f"💾 Saved:\n• Image: {img_path.name}\n• Hist : {hist_path.name}\n• Sum  : {txt_path.name}")
    except Exception as e:
        ui_log(f"⚠️ Save failed: {e}")

def on_show_hist(_=None):
    folder = Path(root_w.value.strip().strip('"'))
    fname = save_name_w.value.strip()
    try:
        show_hist_from_saved(folder, fname)
    except Exception as e:
        ui_log(f"⚠️ {e}")

def apply_bg_from_widgets():
    STATE.bg_enabled = bool(bg_on_w.value)
    STATE.bg_use_renorm = bool(bg_renorm_w.value)
    STATE.bg_renorm_p_lo = float(bg_p_lo_w.value)
    STATE.bg_renorm_p_hi = float(bg_p_hi_w.value)


try:
    bg_cap_btn.on_click(capture_background_from_camera, remove=True)
except Exception:
    pass
bg_cap_btn.on_click(capture_background_from_camera)


In [ ]:
# ---- Binding (без дублей) ----
for btn, fn in [
    (preload_btn, preload_video),
    (fake_prev_btn, start_fake_preview),
    (real_prev_btn, start_real_preview),
    (det_init_btn, init_detector),
    (det_toggle_btn, toggle_detector_btn),
    (save_btn, on_save_frame),
    (show_hist_btn, on_show_hist),
    (start_rec_btn, start_recording),
    (setdef_btn, set_default_crop),
    (full_btn, set_full_crop),
]:
    try:
        btn.on_click(fn, remove=True)
    except Exception:
        pass
    btn.on_click(fn)

# observe crop sliders
for s in (x1_w, x2_w, y1_w, y2_w):
    try:
        s.unobserve_all()
    except Exception:
        pass
    s.observe(lambda c: update_crop_preview(), names="value")

# observe finish sliders
for s in (finish_on_w, fx0_w, fy0_w, fx1_w, fy1_w, fwidth_w):
    try:
        s.unobserve_all()
    except Exception:
        pass
    s.observe(lambda c: update_crop_preview(), names="value")

# observe favorite controls
for s in (fav_enable_w, fav_id_w):
    try:
        s.unobserve_all()
    except Exception:
        pass
    s.observe(lambda c: update_crop_preview(), names="value")

# observe detector checkbox
try:
    det_enable_w.unobserve_all()
except Exception:
    pass
det_enable_w.observe(lambda c: apply_detector_enable(), names="value")

for s in (bg_on_w, bg_n_w, bg_renorm_w, bg_p_lo_w, bg_p_hi_w):
    try:
        s.unobserve_all()
    except Exception:
        pass
    s.observe(lambda c: update_crop_preview(), names="value")


ui_log("✅ UI handlers attached")

if STATE.bg_enabled:
    if STATE.bg_mean is None:
        ui_log("⚠️ BG subtract ON, but background not captured yet")
    else:
        ui_log(f"BG subtract ON | N={STATE.bg_nframes} | {STATE.bg_timestamp}")
else:
    ui_log("BG subtract OFF")


display(
    widgets.VBox([
        widgets.HBox([x1_w, x2_w]),
        widgets.HBox([y1_w, y2_w]),
        widgets.HBox([setdef_btn, full_btn]),

        widgets.HBox([fps_w, exposure_w, duration_w, laser_w]),
        root_w,
        sample_w,
        widgets.HTML("<b>AVI recording</b>"),
        widgets.HBox([avi_on_w, avi_sec_w]),
        widgets.HBox([avi_name_w, avi_fourcc_w, avi_fps_w]),
        widgets.HBox([start_rec_btn, real_prev_btn]),
        widgets.HBox([save_name_w]),
        widgets.HBox([save_btn, show_hist_btn]),

        widgets.HBox([use_fake_w, loop_fake_w, colorize_w, preload_btn, fake_prev_btn]),
        video_path_w,

        widgets.HTML("<b>Finish zone</b>"),
        widgets.HBox([finish_on_w, fwidth_w]),
        widgets.HBox([fx0_w, fy0_w]),
        widgets.HBox([fx1_w, fy1_w]),

        widgets.HTML("<b>Favorite bubble</b>"),
        widgets.HBox([fav_enable_w, fav_id_w]),

        widgets.HTML("<b>Detector</b>"),
        widgets.HBox([det_enable_w, det_warmup_w, det_backend_w, det_gpu_down_w]),
        widgets.HBox([det_ds_w, det_peak_ds_w, det_pct_w, det_min_dist_w, det_radius_w]),
        widgets.HBox([det_thr_mode_w, det_thr_every_w, det_thr_alpha_w, det_stride_w]),
        widgets.HBox([det_init_btn, det_toggle_btn]),

        widgets.HTML("<b>Background</b>"),
        widgets.HBox([bg_on_w, bg_n_w, bg_cap_btn]),
        widgets.HBox([bg_renorm_w, bg_p_lo_w, bg_p_hi_w]),


        out
    ])
)

# initial preview in MPL
update_crop_preview()


In [ ]:
%matplotlib widget

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 100

fig, ax = plt.subplots(figsize=(9, 6))

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Number of variants / objects per experiment (N)")
ax.set_ylabel("Phenotype readout time / required observation window (s)")

# Approximate “regions” as labeled points (можно заменить эллипсами/полигонами)
points = [
    ("Microplates\n(96–1536)",      1e3, 1e2),
    ("FACS / Flow cytometry",       1e6, 1e-2),
    ("Droplet FADS\n(point)",       1e6, 1e-3),
    ("Droplet arrays / traps",      3e4, 3e3),
    ("Raman / MS / NMR\n(assays)",  1e3, 1e3),
]

for label, x, y in points:
    ax.scatter([x], [y])
    ax.text(x*1.1, y*1.1, label, fontsize=9, va="bottom")

# Your platform highlighted
ax.scatter([1e3], [5e2], s=200, marker="*", label="This work")
ax.text(1e3*1.15, 5e2*1.15, "This work:\nwide-field chamber +\npost hoc selection", fontsize=10)

# “problem zone” shading
import numpy as np
x = np.array([1e4, 1e6, 1e6, 1e4])
y = np.array([1e2, 1e2, 1e4, 1e4])
ax.fill(x, y, alpha=0.08)
ax.text(2e4, 2e3, "Slow-readout + large N\n(phenotyping bottleneck)", fontsize=10)

ax.set_xlim(1e2, 1e6)
ax.set_ylim(1e-4, 1e4)

ax.grid(True, which="both", alpha=0.25)
ax.legend(loc="lower left")
plt.xlim(500, 1200000)
plt.ylim(10, 10000)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_screening_map(
    points,
    *,
    P_list=(1, 10, 100, 1000),
    T_levels_hours=(0.5, 1, 4, 24, 24*7),   # изолинии по общему времени, в часах
    N_range=(1e2, 1e10),
    t_range=(1e-3, 1e6),                    # 1 ms .. ~11.6 days
    title="Library size vs per-variant measurement time",
    save_path=None,
):
    """
    Карта: X=N (size of library), Y=t (seconds per variant).
    Изолинии: T_total = (N * t) / P.

    points: list of dicts with keys:
        - label: str
        - N: float (library size)
        - t_s: float (seconds per variant)
        - note: optional str (extra line under label)
    """

    fig, ax = plt.subplots(figsize=(9, 6))

    # Axes settings
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(N_range)
    ax.set_ylim(t_range)
    ax.set_xlabel("Library size, N (number of variants)")
    ax.set_ylabel("Per-variant measurement time, t (s)")
    ax.set_title(title)

    # Make a grid for contour-like isolines (but we'll plot analytic lines for clarity)
    N_vals = np.logspace(np.log10(N_range[0]), np.log10(N_range[1]), 400)

    # Plot isolines for each P and each total-time level
    # Equation: t = (T_total * P) / N
    for P in P_list:
        for Th in T_levels_hours:
            Tsec = Th * 3600.0
            t_line = (Tsec * P) / N_vals
            # keep only visible part
            m = (t_line >= t_range[0]) & (t_line <= t_range[1])
            if np.any(m):
                ax.plot(N_vals[m], t_line[m], linewidth=1)

        # annotate one reference label per P (choose e.g. T=24h line position)
        Th_ref = 24
        Tsec_ref = Th_ref * 3600.0
        t_ref = (Tsec_ref * P) / N_vals
        mref = (t_ref >= t_range[0]) & (t_ref <= t_range[1])
        if np.any(mref):
            # pick a point roughly in the middle of visible segment
            idx = np.where(mref)[0][len(np.where(mref)[0]) // 2]
            ax.text(
                N_vals[idx],
                t_ref[idx],
                f"P={P}",
                fontsize=9,
                ha="left",
                va="bottom",
            )

    # Add a legend-like text block for total time levels
    levels_txt = ", ".join([f"{h:g}h" if h < 24 else (f"{h/24:g}d" if h < 24*7 else f"{h/(24*7):g}w")
                            for h in T_levels_hours])
    ax.text(
        0.02, 0.02,
        "Isolines: constant total screening time\n"
        f"T_levels = {levels_txt}\n"
        "Formula: T_total = (N · t) / P",
        transform=ax.transAxes,
        fontsize=9,
        va="bottom",
        ha="left",
        bbox=dict(boxstyle="round,pad=0.3", alpha=0.15)
    )

    # Plot points + labels
    for p in points:
        N = float(p["N"])
        t_s = float(p["t_s"])
        label = str(p["label"])
        note = str(p.get("note", ""))

        ax.scatter([N], [t_s], s=35)  # default color cycle
        text = label if not note else f"{label}\n{note}"
        ax.text(N * 1.06, t_s * 1.06, text, fontsize=9, ha="left", va="bottom")

    ax.grid(True, which="both", linewidth=0.5, alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=300)
        print("Saved:", save_path)

    return fig, ax


# ----------------------------
# Пример точек (поменяй под себя)
# ----------------------------
points = [
    # Быстрые потоковые
    dict(label="FACS / droplet sorting", N=1e6, t_s=1e-4, note="~10 kHz equiv."),
    dict(label="Image-activated sorting", N=1e5, t_s=1e-2, note="slower, imaging"),

    # Планшеты / классика
    dict(label="96-well assay", N=96, t_s=60, note="~1 min readout"),
    dict(label="384-well assay", N=384, t_s=60, note="~1 min readout"),

    # Медленные фенотипы
    dict(label="ODMR / MFE (single)", N=1, t_s=60, note="minutes per condition"),
    dict(label="Your wide-field chamber", N=1e3, t_s=60, note="parallel slow readout"),
    dict(label="NMR (per sample)", N=1e2, t_s=3600, note="~1 h / sample"),
]

# Запуск (сохрани PNG)
plot_screening_map(
    points,
    P_list=(1, 10, 100, 1000),
    T_levels_hours=(1, 4, 24, 24*7),
    N_range=(1e2, 1e9),
    t_range=(1e-4, 1e5),
    title="Problem map: library size vs measurement time",
    save_path="problem_map.png",
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

# ---------------------------
# Config (slide-friendly)
# ---------------------------
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 100,
    "font.size": 12,
})

def _outlined_text_kwargs():
    # white text with black outline (readable on busy backgrounds)
    return dict(
        path_effects=[pe.withStroke(linewidth=4, foreground="white")]
    )

def plot_problem_map(
    out_png="problem_map_clean.png",
    P_ref=1000,  # reference parallelism for isolines (e.g. droplets measured in parallel)
):
    """
    Conceptual problem map: Library size N vs per-variant measurement time t (seconds).
    Isolines: constant total screening time T_total for a fixed parallelism P_ref:
        T_total = (N * t) / P_ref  =>  t = (T_total * P_ref) / N

    Adjust points/labels manually below.
    """

    fig, ax = plt.subplots(figsize=(13, 7))

    # axes in log scale
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(1e2, 1e9)
    ax.set_ylim(1e-4, 1e5)

    ax.set_title("Problem map: library size vs measurement time (conceptual)")
    ax.set_xlabel("Library size, N (variants)")
    ax.set_ylabel("Per-variant measurement time, t (s)")

    # light grid (not too dense)
    ax.grid(True, which="major", alpha=0.25)
    ax.grid(True, which="minor", alpha=0.10)

    # ---------------------------
    # Isolines of constant total screening time for fixed P_ref
    # ---------------------------
    N = np.logspace(2, 9, 500)

    # choose FEW levels only (slide!)
    T_levels = [
        ("1 h",   3600.0),
        ("4 h",   4 * 3600.0),
        ("1 day", 24 * 3600.0),
        ("1 week", 7 * 24 * 3600.0),
    ]

    # plot isolines
    for label, T in T_levels:
        t = (T * P_ref) / N
        ax.plot(N, t, linewidth=2, alpha=0.65)

    # label isolines at right side (avoid clutter)
    # choose a label anchor near the right edge
    N_label = 7e8
    for label, T in T_levels:
        t_label = (T * P_ref) / N_label
        if ax.get_ylim()[0] < t_label < ax.get_ylim()[1]:
            ax.text(
                N_label, t_label, f"T={label}  (P={P_ref})",
                ha="right", va="bottom",
                fontsize=11,
                **_outlined_text_kwargs()
            )

    # ---------------------------
    # Points (edit these freely)
    # ---------------------------
    # Each point: name, N, t, style
    points = [
        # name, N, t(s), marker, size
        ("NMR (per sample)\n~1 h / sample",           2e2,   3600.0,   "o", 120),
        ("96-well assay\n~1 min / well",              1e3,   60.0,     "o", 120),
        ("Image-activated sorting\n(~10 ms imaging)", 1e5,   1e-2,     "o", 120),
        ("FACS / droplet sorting\n(~10 kHz eq.)",      1e6,   1e-4,     "o", 120),

        # YOUR PLATFORM (примерно)
        ("This work:\nwide-field chamber\nslow readout × parallel", 2e3,  60.0, "*", 260),
    ]

    # manual label offsets in *axes fraction* units (dx, dy)
    # tweak these until it looks perfect on your slide
    label_offsets = {
        "NMR (per sample)\n~1 h / sample":           (-0.08,  0.05),
        "96-well assay\n~1 min / well":              (-0.02,  0.08),
        "Image-activated sorting\n(~10 ms imaging)": ( 0.08,  0.05),
        "FACS / droplet sorting\n(~10 kHz eq.)":     ( 0.08, -0.08),
        "This work:\nwide-field chamber\nslow readout × parallel":  (-0.10, -0.12),
    }

    # draw points + callouts
    for name, n, t, mk, ms in points:
        ax.scatter([n], [t], marker=mk, s=ms, zorder=5)

        # callout box position in data coords (computed via axes-fraction shift)
        dx_af, dy_af = label_offsets.get(name, (0.05, 0.05))

        # Convert (n,t) to axes fraction
        x_af, y_af = ax.transAxes.inverted().transform(ax.transData.transform((n, t)))
        x2_af = np.clip(x_af + dx_af, 0.02, 0.98)
        y2_af = np.clip(y_af + dy_af, 0.02, 0.98)

        # Back to data coords
        x2, y2 = ax.transData.inverted().transform(ax.transAxes.transform((x2_af, y2_af)))

        ax.annotate(
            name,
            xy=(n, t),
            xytext=(x2, y2),
            textcoords="data",
            ha="left", va="center",
            fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85),
            arrowprops=dict(arrowstyle="-", lw=2, color="black"),
            zorder=6,
        )

    # small legend note (formula)
    ax.text(
        0.02, 0.03,
        "Isolines: constant total screening time\n"
        "Formula:  T_total = (N · t) / P",
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="none", alpha=0.75),
    )

    plt.tight_layout()
    plt.savefig(out_png, dpi=100)
    plt.show()
    return out_png


if __name__ == "__main__":
    plot_problem_map(out_png="problem_map_clean.png", P_ref=1000)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# 1) Точки (системы) — порядок величин
#    ВАЖНО: это иллюстративные значения. Подставь свои.
# ---------------------------
points = [
    # label, N_variants, t_seconds_per_variant, (dx,dy) offset for label in points coords (log-scale friendly)
    ("MFE/ODMR-sensitive proteins\n(directed evolution target)", 1e6, 30, (10, 10)),
    ("Flavin photophysics / slow kinetics\n(single droplet time-traces)", 1e4, 60, (10, -15)),
    ("Aptamer candidates\n(kinetics/affinity characterization)", 1e4, 300, (10, 10)),
    ("Ribozymes / riboswitches\n(functional assay)", 1e5, 600, (10, 10)),
    ("Photocatalyst libraries\n(activity/selectivity)", 1e3, 900, (10, 10)),
    ("EPR spin probes / spin chemistry\n(spectra / relaxation)", 1e3, 1800, (10, 10)),
    ("NMR of compound variants\n(per-sample spectra)", 1e2, 3600, (10, 10)),
]

# ---------------------------
# 2) Изолинии T = N*t (последовательно), для наглядности
# ---------------------------
T_levels = [
    ("1 hour", 3600),
    ("1 day", 86400),
    ("1 week", 7*86400),
    ("1 year", 365*86400),
]

# ---------------------------
# 3) Рисование
# ---------------------------
plt.rcParams.update({"figure.dpi": 100, "font.size": 12})

fig, ax = plt.subplots(figsize=(12, 7))

ax.set_xscale("log")
ax.set_yscale("log")

# Диапазоны — подгони под свои задачи
x_min, x_max = 1e2, 1e10
y_min, y_max = 1e-1, 1e5
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

# Изолинии T = N*t => t = T/N
xs = np.logspace(np.log10(x_min), np.log10(x_max), 400)
for name, T in T_levels:
    ys = T / xs
    ax.plot(xs, ys, linestyle="--", linewidth=1.5, alpha=0.55)
    # подпись линии — ставим примерно в “чистой” зоне справа
    x_lab = 3e7
    y_lab = T / x_lab
    if y_min < y_lab < y_max:
        ax.text(x_lab, y_lab, f"T = {name}", fontsize=11, alpha=0.8)

# Точки систем
for label, N, t, (dx, dy) in points:
    ax.scatter([N], [t], s=90, zorder=3)
    ax.annotate(
        label,
        xy=(N, t),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.9),
        arrowprops=dict(arrowstyle="-", lw=0.8, alpha=0.8),
    )

# Лёгкая подсветка “тяжёлой области” (большие N и большие t)
# Это чисто презентационная штука — чтобы глазом сразу читалось.
hard_x0, hard_x1 = 1e5, x_max
hard_y0, hard_y1 = 10, y_max
ax.fill_between([hard_x0, hard_x1], [hard_y0, hard_y0], [hard_y1, hard_y1], alpha=0.08)
ax.text(2e5, 2e4, "Hard regime:\nlarge N + slow per-variant readout",
        fontsize=12, alpha=0.8)

ax.grid(True, which="both", alpha=0.25)
ax.set_title("Problem map: number of variants vs per-variant measurement time")
ax.set_xlabel("Variant space size, N (number of objects / variants)")
ax.set_ylabel("Per-variant measurement time, t (s)")

plt.tight_layout()

# Экспорт: PNG + SVG (в BioRender обычно удобен SVG)
plt.savefig("problem_map_systems.png", dpi=100)
plt.savefig("problem_map_systems.svg")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Global style ---
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 100,
    "font.size": 12,
})

def plot_problem_map_clean(
    *,
    out_png="problem_map_clean.png",
    show_isolines=True,
    isoline_total_times_s=(3600, 4*3600, 24*3600, 7*24*3600, 30*24*3600, 365*24*3600),  # 1h,4h,1d,1w,1mo,1y
    parallelism_P=1,  # assumed parallel channels; keep 1 for the conceptual map
):
    """
    log-log map:
      x = variant space size N
      y = per-variant measurement time t (s)
    isolines: T_total = (N * t) / P
    """

    # ---- Points: (name, N, t_seconds, color, marker) ----
    # Values are approximate placeholders consistent with your example plot.
    # Tweak N and t if you want a different placement.
    points = [
        ("NMR of compound variants",                 1e2,  3.0e3,  "#e377c2", "o"),
        ("EPR spin probes / spin chemistry",         1e3,  9.0e2,  "#9467bd", "o"),
        ("Photocatalysis libraries",                 1e3,  1.6e3,  "#8c564b", "o"),
        ("Aptamer candidates",                       1e4,  3.0e2,  "#2ca02c", "o"),
        ("Ribozymes / riboswitches",                 1e5,  8.0e2,  "#d62728", "o"),
        ("Flavin photophysics / slow kinetics",      1e4,  6.0e1,  "#ff7f0e", "o"),
        ("MFE/ODMR-sensitive proteins",              1e6,  3.0e1,  "#1f77b4", "o"),
    ]

    # ---- Figure ----
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlim(90, 1e9)
    ax.set_ylim(1e-1, 1e5)

    ax.set_xlabel("Variant space size, N (number of objects / variants)")
    ax.set_ylabel("Per-variant measurement time, t (s)")
    ax.set_title("Problem map: number of variants vs per-variant measurement time")

    ax.grid(True, which="both", linestyle="-", alpha=0.15)

    # ---- Isolines: T_total = (N * t) / P ----
    if show_isolines:
        N_grid = np.logspace(2, 10, 400)
        for T in isoline_total_times_s:
            t_line = (T * parallelism_P) / N_grid
            # only plot visible segment
            m = (t_line >= ax.get_ylim()[0]) & (t_line <= ax.get_ylim()[1])
            if np.any(m):
                ax.plot(N_grid[m], t_line[m], linestyle="--", linewidth=2, alpha=0.35)

    # ---- Points (no labels on plot) ----
    for (name, N, t, color, marker) in points:
        ax.scatter([N], [t], s=140, c=[color], marker=marker, edgecolors="black", linewidths=0.6, zorder=5)

    # Optional: legend outside (names only) — you may want to turn it OFF for BioRender
    # Uncomment if you want a legend version.
    # handles = []
    # labels = []
    # for (name, N, t, color, marker) in points:
    #     h = ax.scatter([], [], s=140, c=[color], marker=marker, edgecolors="black", linewidths=0.6)
    #     handles.append(h)
    #     labels.append(name)
    # ax.legend(handles, labels, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=True)

    plt.tight_layout()
    fig.savefig(out_png, dpi=100)
    plt.show()

    return points

points = plot_problem_map_clean(out_png="problem_map_clean.png", show_isolines=True, parallelism_P=1)
print("Saved:", "problem_map_clean.png")
print("Points (for BioRender labels):")
for p in points:
    print(p[0], "| N=", p[1], "| t=", p[2], "s")